# Exercise 2.4: GCN and GraphSAGE on the Warsaw Bike-Sharing Graph

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfeng-hsm/KI_Geodatenanalyse_SS26/blob/main/lectures/02_deep_learning/notebooks/exercise_2_4_warsaw_gcn_graphsage_spatial_baselines.ipynb)

This notebook compares feature-only models with graph neural networks on bike-sharing station demand.

- Data source: Warsaw Bike-Sharing Daily Periods Graph Dataset for GNN, Season 2023, Mendeley Data, DOI: 10.17632/kzvdgfzk4w.1.
- Task: predict station-level trip intensity from station, weather, time, and spatial graph context.
- Models: Random Forest, XGBoost, GCN, GraphSAGE, and GraphSAGE with shuffled edges.
- Main question: when does the graph add useful spatial information beyond ordinary tabular features?

## 1. Setup

The GNN models are implemented directly in PyTorch. This keeps the notebook light and avoids a PyTorch Geometric installation step.

In [ ]:
# Colab setup. In a local environment, run this only if a package is missing.
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    !pip -q install networkx scikit-learn xgboost matplotlib pandas

In [ ]:
from pathlib import Path
import math
import pickle
import random
import urllib.request
import warnings
import zipfile

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import torch
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

## 2. Load Warsaw graph snapshots

Use real `.pt` graph snapshots from the Warsaw dataset. The notebook downloads the course copy from Seafile and reads multiple graphs from the archive.

Each graph file is about 1.3 MB and contains roughly 295 stations. The default setting uses several `afternoon_peak_*.pt` snapshots, not just one file.

The archive is about 340 MB. `MAX_GRAPHS` controls how many snapshots are loaded for the exercise.

In [ ]:
GRAPH_PATTERN = "afternoon_peak_*.pt"
MAX_GRAPHS = 12
DATA_DIR = Path("/content/warsaw_gnn") if IN_COLAB else Path("data/warsaw_gnn")
ARCHIVE_FILE = DATA_DIR / "warsaw_gnn_dataset.zip"
SEAFILE_ARCHIVE_URL = "https://seafile.rlp.net/seafhttp/f/d0d006cf506b4ed79da8/?op=view"
EXAMPLE_GRAPH_FILENAME = "afternoon_peak_01_06_2023.pt"
EXAMPLE_GRAPH_SIZE_BYTES = 1_319_004

def ensure_archive() -> Path:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    if not ARCHIVE_FILE.exists():
        print(f"Downloading Warsaw GNN archive from Seafile to {ARCHIVE_FILE}...")
        urllib.request.urlretrieve(SEAFILE_ARCHIVE_URL, ARCHIVE_FILE)
    else:
        print(f"Using cached archive: {ARCHIVE_FILE}")
    return ARCHIVE_FILE

def graph_names_from_archive(archive_file: Path, pattern: str = GRAPH_PATTERN, max_graphs: int = MAX_GRAPHS) -> list[str]:
    with zipfile.ZipFile(ARCHIVE_FILE) as zf:
        matches = sorted(
            name for name in zf.namelist()
            if Path(name).match(pattern) or Path(name).name == EXAMPLE_GRAPH_FILENAME
        )
        if not matches:
            raise FileNotFoundError(f"No graph files matching {pattern!r} were found in {archive_file}")
        return matches[:max_graphs]

def load_graphs_from_archive() -> list[tuple[str, nx.DiGraph]]:
    archive_file = ensure_archive()
    selected_names = graph_names_from_archive(archive_file)
    graphs = []
    with zipfile.ZipFile(archive_file) as zf:
        for name in selected_names:
            with zf.open(name) as f:
                graph = pickle.load(f)
            if not isinstance(graph, nx.Graph):
                raise TypeError(f"Expected a NetworkX graph in {name}, got {type(graph)!r}")
            if Path(name).name == EXAMPLE_GRAPH_FILENAME:
                info = zf.getinfo(name)
                if info.file_size != EXAMPLE_GRAPH_SIZE_BYTES:
                    raise ValueError(f"Unexpected example graph size: {info.file_size:,} bytes")
            graphs.append((Path(name).name, nx.DiGraph(graph)))
    return graphs

graphs = load_graphs_from_archive()
G = graphs[0][1]
print(f"loaded_graphs={len(graphs)}, station_time_rows={sum(g.number_of_nodes() for _, g in graphs):,}")
print(f"first_graph={graphs[0][0]}, nodes={G.number_of_nodes():,}, edges={G.number_of_edges():,}")

## 3. Convert graph attributes into a node table

The target is station-level trip intensity: incoming plus outgoing `trips_count`, transformed with `log1p`. The feature-only models see only node attributes. The GNNs see the same node attributes plus the graph edges.

In [ ]:
def edge_weight(edge_data: dict) -> float:
    for key in ["trips_count", "trip_count", "count", "weight"]:
        if key in edge_data:
            try:
                return float(edge_data[key])
            except Exception:
                return 0.0
    return 1.0

def graph_to_node_frame(graph: nx.DiGraph, graph_file: str) -> pd.DataFrame:
    total_flow = {node: 0.0 for node in graph.nodes}
    in_flow = {node: 0.0 for node in graph.nodes}
    out_flow = {node: 0.0 for node in graph.nodes}
    for u, v, data in graph.edges(data=True):
        w = edge_weight(data)
        out_flow[u] += w
        in_flow[v] += w
        total_flow[u] += w
        total_flow[v] += w

    rows = []
    for node, attrs in graph.nodes(data=True):
        row = {"graph_file": graph_file, "node_id": node, **dict(attrs)}
        row["target_total_trips"] = total_flow[node]
        row["target_in_trips"] = in_flow[node]
        row["target_out_trips"] = out_flow[node]
        rows.append(row)
    return pd.DataFrame(rows)

nodes = pd.concat(
    [graph_to_node_frame(graph, graph_file) for graph_file, graph in graphs],
    ignore_index=True,
)
nodes["y_log_total_trips"] = np.log1p(nodes["target_total_trips"].astype(float))

exclude_exact = {"graph_file", "node_id", "target_total_trips", "target_in_trips", "target_out_trips", "y_log_total_trips"}
exclude_contains = ["trip", "flow", "target"]
candidate_features = []
for column in nodes.columns:
    if column in exclude_exact or any(term in column.lower() for term in exclude_contains):
        continue
    numeric = pd.to_numeric(nodes[column], errors="coerce")
    if numeric.notna().sum() >= max(5, int(0.2 * len(nodes))) and numeric.nunique(dropna=True) > 1:
        nodes[column] = numeric
        candidate_features.append(column)

if not candidate_features:
    raise ValueError("No numeric node features were found. Inspect the node attributes and select features manually.")

feature_frame = nodes[candidate_features].copy()
feature_frame = feature_frame.replace([np.inf, -np.inf], np.nan)
feature_frame = feature_frame.fillna(feature_frame.median(numeric_only=True)).fillna(0)

print(f"Using {len(candidate_features)} node features across {nodes['graph_file'].nunique()} graph snapshots:")
print(candidate_features)
nodes[["graph_file", "node_id", "target_total_trips", "y_log_total_trips"] + candidate_features[:8]].head()

In [ ]:
edges = pd.concat(
    [
        pd.DataFrame(
            [{"graph_file": graph_file, "source": u, "target": v, "trips_count": edge_weight(data)} for u, v, data in graph.edges(data=True)]
        )
        for graph_file, graph in graphs
    ],
    ignore_index=True,
)
display(edges.head())

fig, ax = plt.subplots(figsize=(7, 4))
nodes["target_total_trips"].hist(ax=ax, bins=30, color="#3f7f93", edgecolor="white")
ax.set_title("Station trip intensity distribution")
ax.set_xlabel("incoming + outgoing trips")
ax.set_ylabel("stations")
plt.show()

## 4. Train/test split and feature-only baselines

Random Forest and XGBoost use the same node table but no graph edges. This is the ordinary tabular baseline.

In [ ]:
X = feature_frame.to_numpy(dtype=np.float32)
y = nodes["y_log_total_trips"].to_numpy(dtype=np.float32)

graph_files = np.array(sorted(nodes["graph_file"].unique()))
train_graphs, test_graphs = train_test_split(
    graph_files, test_size=0.25, random_state=SEED
)
train_graphs, val_graphs = train_test_split(
    train_graphs, test_size=0.25, random_state=SEED
)
train_idx = nodes.index[nodes["graph_file"].isin(train_graphs)].to_numpy()
val_idx = nodes.index[nodes["graph_file"].isin(val_graphs)].to_numpy()
test_idx = nodes.index[nodes["graph_file"].isin(test_graphs)].to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit(X[train_idx]).transform(X).astype(np.float32)
y_mean = float(y[train_idx].mean())
y_std = float(y[train_idx].std() + 1e-8)
print(f"train_graphs={len(train_graphs)}, validation_graphs={len(val_graphs)}, test_graphs={len(test_graphs)}")
print(f"train_rows={len(train_idx)}, validation_rows={len(val_idx)}, test_rows={len(test_idx)}")

def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mse = mean_squared_error(y_true, y_pred)
    return {
        "rmse": float(math.sqrt(mse)),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }

results = []
predictions = {}

rf = RandomForestRegressor(n_estimators=350, min_samples_leaf=3, random_state=SEED, n_jobs=-1)
rf.fit(X_scaled[train_idx], y[train_idx])
pred_rf = rf.predict(X_scaled[test_idx])
results.append({"model": "Random Forest", **regression_metrics(y[test_idx], pred_rf)})
predictions["Random Forest"] = pred_rf

try:
    from xgboost import XGBRegressor

    xgb = XGBRegressor(
        n_estimators=400,
        max_depth=4,
        learning_rate=0.035,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=SEED,
    )
    xgb.fit(X_scaled[train_idx], y[train_idx])
    pred_xgb = xgb.predict(X_scaled[test_idx])
    results.append({"model": "XGBoost", **regression_metrics(y[test_idx], pred_xgb)})
    predictions["XGBoost"] = pred_xgb
except Exception as exc:
    print(f"XGBoost unavailable ({exc}). Using sklearn HistGradientBoostingRegressor as a fallback.")
    from sklearn.ensemble import HistGradientBoostingRegressor

    hgb = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.035, random_state=SEED)
    hgb.fit(X_scaled[train_idx], y[train_idx])
    pred_hgb = hgb.predict(X_scaled[test_idx])
    results.append({"model": "HistGradientBoosting", **regression_metrics(y[test_idx], pred_hgb)})
    predictions["HistGradientBoosting"] = pred_hgb

pd.DataFrame(results).sort_values("rmse")

## 5. Build graph tensors

The GNNs receive an adjacency matrix from the station graph. We also create a shuffled-edge graph as a negative control. If the real graph is useful, the real-edge GNN should beat the shuffled-edge version.

In [ ]:
node_order = list(zip(nodes["graph_file"], nodes["node_id"]))
node_to_pos = {node_key: i for i, node_key in enumerate(node_order)}

edge_pairs = []
for graph_file, graph in graphs:
    for u, v in graph.edges():
        source_key = (graph_file, u)
        target_key = (graph_file, v)
        if source_key in node_to_pos and target_key in node_to_pos:
            edge_pairs.append((node_to_pos[source_key], node_to_pos[target_key]))

if not edge_pairs:
    raise ValueError("The graph has no usable edges after node alignment.")

def make_undirected_edges(edge_pairs: list[tuple[int, int]]) -> list[tuple[int, int]]:
    undirected = set()
    for u, v in edge_pairs:
        if u == v:
            continue
        undirected.add((u, v))
        undirected.add((v, u))
    return sorted(undirected)

def shuffled_edges(edge_pairs: list[tuple[int, int]], n_nodes: int, seed: int = SEED) -> list[tuple[int, int]]:
    rng = np.random.default_rng(seed)
    perm = rng.permutation(n_nodes)
    shuffled = [(int(perm[u]), int(perm[v])) for u, v in edge_pairs if perm[u] != perm[v]]
    return shuffled

def row_normalized_adjacency(edge_pairs: list[tuple[int, int]], n_nodes: int, add_self: bool) -> torch.Tensor:
    A = torch.zeros((n_nodes, n_nodes), dtype=torch.float32)
    for u, v in edge_pairs:
        A[v, u] = 1.0
    if add_self:
        A += torch.eye(n_nodes, dtype=torch.float32)
    degree = A.sum(dim=1, keepdim=True).clamp(min=1.0)
    return A / degree

n = len(nodes)
real_edges = make_undirected_edges(edge_pairs)
random_edges = make_undirected_edges(shuffled_edges(edge_pairs, n))
adj_mean = row_normalized_adjacency(real_edges, n, add_self=False).to(DEVICE)
adj_gcn = row_normalized_adjacency(real_edges, n, add_self=True).to(DEVICE)
adj_mean_shuffled = row_normalized_adjacency(random_edges, n, add_self=False).to(DEVICE)

X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=DEVICE)
y_scaled = ((y - y_mean) / y_std).astype(np.float32)
y_tensor = torch.tensor(y_scaled.reshape(-1, 1), dtype=torch.float32, device=DEVICE)
train_tensor = torch.tensor(train_idx, dtype=torch.long, device=DEVICE)
val_tensor = torch.tensor(val_idx, dtype=torch.long, device=DEVICE)
test_tensor = torch.tensor(test_idx, dtype=torch.long, device=DEVICE)

print(f"real undirected edges={len(real_edges):,}, shuffled undirected edges={len(random_edges):,}")

## 6. GCN and GraphSAGE in PyTorch

- GCN smooths node representations with the normalized adjacency matrix.
- GraphSAGE concatenates each station's own features with the mean feature vector from neighboring stations.
- The shuffled-edge GraphSAGE model has the same node features and a graph of similar size, but the spatial relations are wrong.

In [ ]:
class GCNRegressor(torch.nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.lin1 = torch.nn.Linear(in_dim, hidden_dim)
        self.lin2 = torch.nn.Linear(hidden_dim, hidden_dim)
        self.out = torch.nn.Linear(hidden_dim, 1)
        self.dropout = torch.nn.Dropout(0.15)

    def forward(self, x: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        h = torch.relu(self.lin1(adj @ x))
        h = self.dropout(h)
        h = torch.relu(self.lin2(adj @ h))
        return self.out(h)

class GraphSAGELayer(torch.nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.lin = torch.nn.Linear(in_dim * 2, out_dim)

    def forward(self, x: torch.Tensor, adj_mean: torch.Tensor) -> torch.Tensor:
        neighbor_mean = adj_mean @ x
        h = torch.cat([x, neighbor_mean], dim=1)
        return torch.relu(self.lin(h))

class GraphSAGERegressor(torch.nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.sage1 = GraphSAGELayer(in_dim, hidden_dim)
        self.sage2 = GraphSAGELayer(hidden_dim, hidden_dim)
        self.out = torch.nn.Linear(hidden_dim, 1)
        self.dropout = torch.nn.Dropout(0.15)

    def forward(self, x: torch.Tensor, adj_mean: torch.Tensor) -> torch.Tensor:
        h = self.sage1(x, adj_mean)
        h = self.dropout(h)
        h = self.sage2(h, adj_mean)
        return self.out(h)

def train_gnn(model: torch.nn.Module, adj: torch.Tensor, epochs: int = 800, lr: float = 0.01, patience: int = 100) -> tuple[np.ndarray, list[float]]:
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = torch.nn.SmoothL1Loss()
    history = []
    best_state = None
    best_val_loss = float("inf")
    stale_epochs = 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        pred = model(X_tensor, adj)
        loss = loss_fn(pred[train_tensor], y_tensor[train_tensor])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(X_tensor, adj)
            val_loss = loss_fn(val_pred[val_tensor], y_tensor[val_tensor])
        history.append(float(val_loss.detach().cpu()))

        if history[-1] < best_val_loss - 1e-5:
            best_val_loss = history[-1]
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            stale_epochs = 0
        else:
            stale_epochs += 1
        if stale_epochs >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        test_pred_scaled = model(X_tensor, adj)[test_tensor].detach().cpu().numpy().ravel()
        test_pred = test_pred_scaled * y_std + y_mean
    return test_pred, history

torch.manual_seed(SEED)
pred_gcn, hist_gcn = train_gnn(GCNRegressor(X_scaled.shape[1]), adj_gcn)
results.append({"model": "GCN", **regression_metrics(y[test_idx], pred_gcn)})
predictions["GCN"] = pred_gcn

torch.manual_seed(SEED)
pred_sage, hist_sage = train_gnn(GraphSAGERegressor(X_scaled.shape[1]), adj_mean)
results.append({"model": "GraphSAGE", **regression_metrics(y[test_idx], pred_sage)})
predictions["GraphSAGE"] = pred_sage

torch.manual_seed(SEED)
pred_sage_shuffled, hist_sage_shuffled = train_gnn(GraphSAGERegressor(X_scaled.shape[1]), adj_mean_shuffled)
results.append({"model": "GraphSAGE shuffled edges", **regression_metrics(y[test_idx], pred_sage_shuffled)})
predictions["GraphSAGE shuffled edges"] = pred_sage_shuffled

pd.DataFrame(results).sort_values("rmse")

## 7. Compare results

Lower RMSE and MAE are better. Higher R2 is better.

In [ ]:
metrics_table = pd.DataFrame(results).sort_values("rmse").reset_index(drop=True)
display(metrics_table)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric, color in zip(axes, ["rmse", "mae", "r2"], ["#507dbc", "#3f7f93", "#8a6f3d"]):
    ordered = metrics_table.sort_values(metric, ascending=(metric != "r2"))
    ax.barh(ordered["model"], ordered[metric], color=color)
    ax.set_title(metric.upper())
    ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
best_model = metrics_table.iloc[0]["model"]
best_pred = predictions[best_model]

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(y[test_idx], best_pred, alpha=0.75, color="#2f6f73", edgecolor="white", linewidth=0.5)
lims = [min(y[test_idx].min(), best_pred.min()), max(y[test_idx].max(), best_pred.max())]
ax.plot(lims, lims, color="#333333", linestyle="--", linewidth=1)
ax.set_xlabel("observed log1p trips")
ax.set_ylabel("predicted log1p trips")
ax.set_title(f"Observed vs predicted: {best_model}")
ax.grid(alpha=0.25)
plt.show()

## 8. Spatial diagnostics

A strong tree model can win because the node table already contains explicit spatial variables such as coordinates and distances to points of interest. Use these diagnostics to separate two questions:

- Does the real graph carry signal? Compare GraphSAGE with real edges against GraphSAGE with shuffled edges.
- Do ordinary feature-only models use spatial information? Remove explicit spatial variables and compare the tabular baselines again.

In [ ]:
score_by_model = metrics_table.set_index("model")
diagnostics = []

if {"GraphSAGE", "GraphSAGE shuffled edges"}.issubset(score_by_model.index):
    real_rmse = score_by_model.loc["GraphSAGE", "rmse"]
    shuffled_rmse = score_by_model.loc["GraphSAGE shuffled edges", "rmse"]
    diagnostics.append(
        {
            "question": "Do real edges matter?",
            "comparison": "GraphSAGE shuffled RMSE - GraphSAGE RMSE",
            "delta_rmse": shuffled_rmse - real_rmse,
            "interpretation": "positive means the real station graph helps",
        }
    )

feature_only = [m for m in ["Random Forest", "XGBoost", "HistGradientBoosting"] if m in score_by_model.index]
gnn_models = [m for m in ["GCN", "GraphSAGE"] if m in score_by_model.index]
if feature_only and gnn_models:
    best_feature_rmse = score_by_model.loc[feature_only, "rmse"].min()
    best_gnn_rmse = score_by_model.loc[gnn_models, "rmse"].min()
    diagnostics.append(
        {
            "question": "Does a GNN beat feature-only models here?",
            "comparison": "best feature-only RMSE - best GNN RMSE",
            "delta_rmse": best_feature_rmse - best_gnn_rmse,
            "interpretation": "positive means a GNN wins; negative means tabular features are stronger",
        }
    )

pd.DataFrame(diagnostics)

In [ ]:
def is_explicit_spatial_feature(column: str) -> bool:
    lower = column.lower()
    spatial_terms = [
        "lat", "latitude", "lng", "lon", "longitude", "centroid", "coord", "x", "y",
        "distance", "dist", "meters", "km",
    ]
    return lower.startswith("d_") or any(term == lower or term in lower for term in spatial_terms)

spatial_feature_columns = [c for c in candidate_features if is_explicit_spatial_feature(c)]
non_spatial_feature_columns = [c for c in candidate_features if c not in spatial_feature_columns]

print(f"Explicit spatial features removed: {len(spatial_feature_columns)}")
print(spatial_feature_columns[:20])
print(f"Remaining non-spatial features: {len(non_spatial_feature_columns)}")

def scaled_feature_matrix(feature_columns: list[str]) -> np.ndarray:
    frame = nodes[feature_columns].replace([np.inf, -np.inf], np.nan)
    frame = frame.fillna(frame.iloc[train_idx].median(numeric_only=True)).fillna(0)
    values = frame.to_numpy(dtype=np.float32)
    return StandardScaler().fit(values[train_idx]).transform(values).astype(np.float32)

def run_tabular_ablation(feature_columns: list[str], label: str) -> list[dict]:
    if not feature_columns:
        return []
    X_variant = scaled_feature_matrix(feature_columns)
    rows = []

    rf_variant = RandomForestRegressor(n_estimators=350, min_samples_leaf=3, random_state=SEED, n_jobs=-1)
    rf_variant.fit(X_variant[train_idx], y[train_idx])
    rows.append({"feature_set": label, "model": "Random Forest", **regression_metrics(y[test_idx], rf_variant.predict(X_variant[test_idx]))})

    try:
        xgb_variant = XGBRegressor(
            n_estimators=400,
            max_depth=4,
            learning_rate=0.035,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="reg:squarederror",
            random_state=SEED,
        )
        xgb_variant.fit(X_variant[train_idx], y[train_idx])
        rows.append({"feature_set": label, "model": "XGBoost", **regression_metrics(y[test_idx], xgb_variant.predict(X_variant[test_idx]))})
    except Exception:
        pass
    return rows

tabular_ablation = []
tabular_ablation.extend(run_tabular_ablation(candidate_features, "all node features"))
tabular_ablation.extend(run_tabular_ablation(non_spatial_feature_columns, "without explicit spatial features"))
tabular_ablation_table = pd.DataFrame(tabular_ablation).sort_values(["feature_set", "rmse"])
display(tabular_ablation_table)

## 9. Spatial view

If latitude and longitude are available, plot stations by target intensity. Spatial clusters are one reason graph models can help.

In [ ]:
lat_candidates = [c for c in nodes.columns if c.lower() in {"lat", "latitude", "y"} or "latitude" in c.lower()]
lng_candidates = [c for c in nodes.columns if c.lower() in {"lng", "lon", "longitude", "x"} or "longitude" in c.lower()]

if lat_candidates and lng_candidates:
    lat_col, lng_col = lat_candidates[0], lng_candidates[0]
    plot_nodes = nodes.groupby("node_id", as_index=False).agg({lat_col: "first", lng_col: "first", "target_total_trips": "mean"})
    plot_nodes[lat_col] = pd.to_numeric(plot_nodes[lat_col], errors="coerce")
    plot_nodes[lng_col] = pd.to_numeric(plot_nodes[lng_col], errors="coerce")
    plot_nodes = plot_nodes.dropna(subset=[lat_col, lng_col])

    fig, ax = plt.subplots(figsize=(7, 6))
    sc = ax.scatter(
        plot_nodes[lng_col],
        plot_nodes[lat_col],
        c=plot_nodes["target_total_trips"],
        cmap="viridis",
        s=35,
        edgecolor="white",
        linewidth=0.4,
    )
    ax.set_title("Station trip intensity in space")
    ax.set_xlabel(lng_col)
    ax.set_ylabel(lat_col)
    plt.colorbar(sc, ax=ax, label="incoming + outgoing trips")
    ax.grid(alpha=0.2)
    plt.show()
else:
    print("No latitude/longitude columns found. Available columns:")
    print(list(nodes.columns))

## 10. Interpretation checklist

- If XGBoost or Random Forest wins on all node features, this does not mean space is irrelevant. It often means explicit spatial covariates such as coordinates and POI distances already encode much of the spatial signal.
- If GraphSAGE beats the shuffled-edge GraphSAGE model, the real station graph carries useful spatial or mobility structure.
- If tabular models get worse after removing explicit spatial variables, ordinary feature-only models were also using spatial information.
- GCN can perform worse than GraphSAGE here because simple adjacency smoothing can over-smooth station features on a small, dense mobility graph.
- Results are more meaningful when several daily-period graphs are used and train/validation/test splits are separated by graph snapshot.

Exercise extension: aggregate several Warsaw graph files into one training set and test whether GNN gains are larger during morning and afternoon peak periods.